Save as: module8-simulation-lab.ipynb

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/moncap/micro-course/blob/main/module-08/module8-simulation-lab.ipynb)

# Lab 8 — Silicon Villagers and the Commons
### Communication as a treatment in a repeated common-pool game
*Microeconomic Theory for Experimentalists & Applied Microeconomists*

**Research question:** Do LLM villagers overextract a common pool the
way theory predicts — and does a costless, non-binding message exchange
raise cooperation the way it did for Ostrom, Walker & Gardner's (1992)
humans? And the planted question: is silicon cooperation *too* good?

**The game (the class's own parameters):** groups of 4; pool starts at
40; each round each villager requests 0–10; over-requests split the
pool proportionally; the remainder grows 1.5×, capped at 40; six
rounds. Benchmarks from the arithmetic: sustainable ≈ 3 per villager
forever (18 each over six rounds); the all-grab pays 10 once; one
deviator against three sustainers earns 13 vs. their 6.

**The treatment:** in `comm` groups, each villager sends one short
message to the group before every round — pure cheap talk, exactly as
non-binding as OWG's. `no_comm` groups decide in silence.

**Benchmarks (three-way):** OWG (1992) — communication raises
cooperation dramatically but *imperfectly* (pledge-breaking, erosion,
endgame defection persist in humans); the class's own two-phase play
from the experiment session (enter it in the analysis cell); the
sustainable/myopic arithmetic.

**Hypotheses to state before running (see handout):** H1 silent groups
overextract and degrade the pool; H2 communication moves extraction
toward sustainable; H3 — commit NOW to what "too clean" would look
like (pledge-keeping rate, final-round behavior) before you see the
results.

**This is the course's first multi-agent, multi-round lab.** The game
engine cell orchestrates groups, rounds, and message exchange; your
modifications stay in the CHANGE HERE cells as always.


## Setup and the DRY_RUN switch

The notebook starts in **DRY_RUN mode**: it executes end-to-end on synthetic
placeholder data, with no API key and no cost, so you can check your design
and see the analysis pipeline work.

> ⚠️ **DRY_RUN output is fake.** It is generated by `mock_response()`, not
> by a model. Never report it as a result.

When your design is ready, set `DRY_RUN = False` in the cell below and run
again in Colab — you will be asked for your Anthropic API key via a hidden
prompt (the key is never stored in the notebook file).


In [ ]:
%pip install anthropic pandas matplotlib --quiet

In [ ]:
import itertools
import json
import random
import re
import time

import pandas as pd

DRY_RUN = True   # <-- set to False in Colab to query the real model

MODEL = "claude-sonnet-5"   # swap models here to test robustness across models
TEMPERATURE = 1.0            # fixed and reported — part of the experimental design
N_PER_CELL = 30              # decisions per persona x treatment cell

if DRY_RUN:
    print("=" * 62)
    print("DRY RUN: synthetic placeholder data — NOT model behavior.")
    print("Set DRY_RUN = False to run the real experiment.")
    print("=" * 62)

    class _MockBlock:
        def __init__(self, text):
            self.text = text

    class _MockResponse:
        def __init__(self, text):
            self.content = [_MockBlock(text)]

    class _MockMessages:
        def create(self, **kwargs):
            # mock_response() is defined in the design cell below
            return _MockResponse(mock_response(kwargs["messages"][0]["content"]))

    class _MockClient:
        messages = _MockMessages()

    client = _MockClient()
else:
    from getpass import getpass

    import anthropic

    # Never hard-code API keys. getpass keeps the key out of the notebook file.
    API_KEY = getpass("Paste your Anthropic API key: ")
    client = anthropic.Anthropic(api_key=API_KEY)


## Game and agent design — CHANGE HERE (all four blocks live in this cell)

2 treatments × 4 groups × 4 villagers × 6 rounds = 192 decisions, plus
one message per villager per round in `comm` groups (+96 calls).

Personas are deliberately plain — villagers who depend on the pool,
with no cooperative or selfish coaching; the treatment must do the
work. Your required modification (handout) will usually mean **(1)**
editing `PERSONAS` (the defector modification), the game constants (the
endgame announcement), or the round flow, **(2)** editing the two
prompt builders *without changing other wording across treatments*,
and **(3)** touching parsers / `mock_response` only if formats change.


In [ ]:
PERSONAS = {                                     # CHANGE HERE (1 of 4)
    # One entry per villager seat (0-3). Identical by default: the
    # TREATMENT, not the persona, should drive any cooperation.
    0: "You are a villager whose household depends on fish from the shared lake.",
    1: "You are a villager whose household depends on fish from the shared lake.",
    2: "You are a villager whose household depends on fish from the shared lake.",
    3: "You are a villager whose household depends on fish from the shared lake.",
}

TREATMENTS = ["no_comm", "comm"]
N_GROUPS = 4          # groups per treatment
GROUP_SIZE = 4
N_ROUNDS = 6
POOL_START, POOL_CAP, GROWTH, MAX_TAKE = 40, 40, 1.5, 10

RULES = (
    f"Your group of {GROUP_SIZE} villagers shares a fish stock. Each "
    f"round, each villager privately requests a catch of 0 to {MAX_TAKE}. "
    "If total requests exceed the stock, the stock is split "
    "proportionally and exhausted. Whatever remains after the catch "
    f"grows by {GROWTH}x, up to a maximum of {POOL_CAP}. The game lasts "
    f"{N_ROUNDS} rounds. Each fish caught is worth $1 to you."
)


def history_text(history: list) -> str:
    if not history:
        return "This is the first round."
    lines = [
        f"  Round {h['round']}: stock was {h['pool_before']:.0f}, total "
        f"caught {h['total_received']:.1f}, your catch {h['own_received']:.1f}"
        for h in history
    ]
    return "Past rounds:\n" + "\n".join(lines)


def build_message_prompt(seat, rnd, pool, history, prior_msgs):  # CHANGE HERE (2 of 4)
    msgs = ("Messages so far this round:\n" +
            "\n".join(f"  Villager {s}: {m}" for s, m in prior_msgs)
            if prior_msgs else "You are the first to speak this round.")
    return (
        f"{PERSONAS[seat]}\n\n{RULES}\n\n"
        f"Round {rnd} of {N_ROUNDS}. Current stock: {pool:.0f}.\n"
        f"{history_text(history)}\n\n{msgs}\n\n"
        "Before deciding, you may send ONE short message (at most 15 "
        "words) to your group. Messages are not binding.\n\n"
        'Respond with ONLY a JSON object: {"message": "<your message>"}.'
    )


def build_decision_prompt(seat, rnd, pool, history, messages):
    msgs = ("This round's messages:\n" +
            "\n".join(f"  Villager {s}: {m}" for s, m in messages)
            if messages else "There is no communication in your group.")
    return (
        f"{PERSONAS[seat]}\n\n{RULES}\n\n"
        f"Round {rnd} of {N_ROUNDS}. Current stock: {pool:.0f}.\n"
        f"{history_text(history)}\n\n{msgs}\n\n"
        "Privately choose your catch request for this round. Decide as "
        "you genuinely would.\n\n"
        'Respond with ONLY a JSON object: {"take": <integer 0-'
        f'{MAX_TAKE}>}}.'
    )


def parse_take(text: str) -> "int | None":       # CHANGE HERE (3 of 4)
    match = re.search(r'\{\s*"take"\s*:\s*(\d+)\s*\}', text)
    if match:
        take = int(match.group(1))
        if 0 <= take <= MAX_TAKE:
            return take
    return None


def parse_message(text: str) -> "str | None":
    match = re.search(r'\{\s*"message"\s*:\s*"([^"]{1,120})"\s*\}', text)
    return match.group(1) if match else None


def mock_response(prompt: str) -> str:           # CHANGE HERE (4 of 4)
    """DRY_RUN only: synthetic placeholder answers so the notebook
    executes without API calls. NOT model behavior — deliberately
    mimics the planted pattern: silent groups escalate toward crash;
    communicating groups pledge and cooperate TOO well (no endgame
    defection) so the divergence-diagnosis discussion has a target."""
    if '"message"' in prompt.split("Respond with ONLY")[-1]:
        return json.dumps({"message": random.choice([
            "Let us each take 3 so the lake lasts.",
            "I will take 3; please do the same.",
            "Keep catches small, the stock must survive.",
            "Agreed, 3 each protects everyone.",
        ])})
    rnd_match = re.search(r"Round (\d+) of", prompt)
    rnd = int(rnd_match.group(1)) if rnd_match else 1
    if "no communication" in prompt:
        center = 5.0 + 0.6 * rnd                 # silent escalation
    else:
        center = 3.0                              # harmonious, all rounds
    take = int(round(min(max(random.gauss(center, 0.9), 0), MAX_TAKE)))
    return json.dumps({"take": take})


## The game engine *(do not modify)*

Runs every treatment × group: in `comm` groups each villager sends one
message (seeing earlier messages that round), then all villagers decide
simultaneously given the pool, the history, and the messages. Handles
proportional rationing and pool growth. Saves `lab8_results.csv` (one
row per villager-round, messages included) and `lab8_prompts_log.json`
(example prompts). With `DRY_RUN = False` this makes ~288 API calls.


In [ ]:
def ask(prompt: str, parser):
    try:
        response = client.messages.create(
            model=MODEL, max_tokens=150, temperature=TEMPERATURE,
            messages=[{"role": "user", "content": prompt}],
        )
        raw = response.content[0].text
        return parser(raw), raw
    except Exception as err:                     # log failures; never drop silently
        time.sleep(5)
        return None, f"ERROR: {err}"


def run_experiment() -> pd.DataFrame:
    rows, example_prompts = [], {}
    for treatment in TREATMENTS:
        for g in range(N_GROUPS):
            pool = float(POOL_START)
            histories = {s: [] for s in range(GROUP_SIZE)}
            for rnd in range(1, N_ROUNDS + 1):
                messages = []
                if treatment == "comm":
                    for seat in range(GROUP_SIZE):
                        mp = build_message_prompt(seat, rnd, pool,
                                                  histories[seat], messages)
                        msg, raw_m = ask(mp, parse_message)
                        messages.append((seat, msg if msg else "(unparsed)"))
                        if treatment not in example_prompts:
                            example_prompts[treatment] = mp
                requests = {}
                for seat in range(GROUP_SIZE):
                    dp = build_decision_prompt(
                        seat, rnd, pool, histories[seat],
                        messages if treatment == "comm" else None)
                    take, raw_d = ask(dp, parse_take)
                    requests[seat] = (take, raw_d)
                    example_prompts.setdefault(f"{treatment}_decision", dp)
                total_req = sum(t for t, _ in requests.values() if t is not None)
                pool_before = pool
                for seat, (take, raw_d) in requests.items():
                    if take is None:
                        received = None
                    elif total_req <= pool:
                        received = float(take)
                    else:
                        received = take * pool / total_req if total_req else 0.0
                    rows.append({
                        "treatment": treatment, "group": g, "round": rnd,
                        "seat": seat, "requested": take, "received": received,
                        "pool_before": pool_before,
                        "messages": " | ".join(m for _, m in messages) if messages else "",
                        "raw": raw_d, "model": MODEL, "temperature": TEMPERATURE,
                    })
                distributed = min(total_req, pool)
                pool = min((pool - distributed) * GROWTH, POOL_CAP)
                for seat in range(GROUP_SIZE):
                    take = requests[seat][0]
                    received = (float(take) if take is not None and total_req <= pool_before
                                else (take * pool_before / total_req
                                      if take is not None and total_req else 0.0))
                    histories[seat].append({
                        "round": rnd, "pool_before": pool_before,
                        "total_received": distributed,
                        "own_received": received if received is not None else 0.0,
                    })
            print(f"  {treatment} group {g}: final pool {pool:.1f}")

    df = pd.DataFrame(rows)
    df.to_csv("lab8_results.csv", index=False)
    with open("lab8_prompts_log.json", "w") as f:
        json.dump(example_prompts, f, indent=2)
    return df


df = run_experiment()


## Analysis vs. the three benchmarks *(do not modify)*

Outputs: extraction paths by treatment, pool survival, efficiency
against the sustainable maximum (18 per villager), the endgame check
(round 6 vs. round 5 — humans defect at known ends), and a message
sample. Enter the class's own two-phase results where indicated.


In [ ]:
SUSTAINABLE_TAKE, SUSTAINABLE_TOTAL = 3, 18   # per villager per round / per game
# Enter the class's experiment-session results here (from the instructor):
CLASS_MEAN_TAKE = {"no_comm": None, "comm": None}

valid = df.dropna(subset=["requested"]).copy()
print(f"Parse-failure rate: {df['requested'].isna().mean():.3f} "
      "(report this in Section 4 of the write-up)")

print("\n=== Mean requested catch by round x treatment ===")
path = valid.pivot_table(index="round", columns="treatment",
                         values="requested", aggfunc="mean").round(2)
print(path)
print(f"Sustainable benchmark: {SUSTAINABLE_TAKE} per villager per round.")

print("\n=== Pool trajectory (mean pool at round start) ===")
pools = valid.pivot_table(index="round", columns="treatment",
                          values="pool_before", aggfunc="mean").round(1)
print(pools)

print("\n=== Efficiency: mean earnings per villager vs sustainable 18 ===")
earn = (valid.groupby(["treatment", "group", "seat"])["received"].sum()
        .groupby("treatment").mean().round(2))
print(earn)
print(f"All-grab benchmark: 10. OWG humans: communication raises "
      "efficiency dramatically — but imperfectly.")

print("\n=== The endgame check (humans defect in known final rounds) ===")
end = valid[valid["round"].isin([N_ROUNDS - 1, N_ROUNDS])].pivot_table(
    index="round", columns="treatment", values="requested", aggfunc="mean").round(2)
print(end)
print("No round-6 uptick in comm groups = the too-clean signature "
      "(PS8 Part C).")

if CLASS_MEAN_TAKE["comm"] is not None:
    print(f"\nClass benchmark: {CLASS_MEAN_TAKE}")
else:
    print("\n(Enter CLASS_MEAN_TAKE above for the three-way comparison.)")

print("\n=== Message sample (comm groups) ===")
sample = valid[(valid["treatment"] == "comm") & (valid["round"] <= 2)]
for m in sample["messages"].dropna().unique()[:4]:
    print(f"  {m}")
print("Are these pledges kept? Compare each group's takes to its "
      "messages — pledge-keeping near 100% is a divergence from humans.")


## Robustness — required in every lab

Rerun at least one group per treatment with:

1. **a paraphrased prompt** (rewrite both builders' wording, same
   content);
2. **a second model** (change `MODEL` in the setup cell);
3. **the message-ablation run** — in the engine's message phase,
   replace each parsed message with content-free filler ("hello")
   before it enters the decision prompts (three lines in the engine;
   flag your edit clearly). If cooperation survives contentless
   messages, the channel's existence — not the covenant — drove your
   result, and "communication works" was recitation, not mechanism.

**Limits of silicon subjects:** (1) *harmony bias* — alignment tuning
favors agreeable, promise-keeping text; honoring a pledge is the
low-loss continuation of a conversation containing one, payoffs be
damned; (2) *recitation* — Ostrom's result is the most famous in this
literature; (3) *no temptation channel* — fish are words to a model, so
restraint costs nothing and nothing erodes. Perfect cooperation is a
divergence wearing a success costume: your write-up's Section 5
diagnoses it, and Section 6 states what only a human experiment — with
real temptation and real endgames — can establish (PS8 Part C is the
exam version).
